# Batch Normalization – Practice

Batch normalization is most useful when building deep neural networks. To demonstrate this, we'll create a convolutional neural network with 20 convolutional layers, followed by a fully connected layer. We'll use it to classify handwritten digits in the MNIST dataset, which should be familiar to you by now.

This is **not** a good network for classfying MNIST digits. You could create a _much_ simpler network and get _better_ results. However, to give you hands-on experience with batch normalization, we had to make an example that was:
1. Complicated enough that training would benefit from batch normalization.
2. Simple enough that it would train quickly, since this is meant to be a short exercise just to give you some practice adding batch normalization.
3. Simple enough that the architecture would be easy to understand without additional resources.

This notebook includes two versions of the network that you can edit. The first uses PyTorch's ready-made batch normalization layers (`nn.BatchNorm1d` and `nn.BatchNorm2d`). The second is the same network, but uses a batch normalization layer you write yourself from basic tensor operations.

1. [Batch Normalization with `nn.BatchNorm`](#example_1)
2. [Batch Normalization with your own layer](#example_2)

The following cell loads PyTorch, downloads the MNIST dataset if necessary, and loads it into an object named `mnist`. You'll need to run this cell before running anything else in the notebook.

In [ ]:
import os

import torch
import torch.nn.functional as F
from torch import nn

import helper

# Use the GPU when there is one; otherwise fall back to the CPU.
device = helper.device
print("Using device:", device)

# Set the environment variable DL_SMOKE=1 to train for far fewer batches (a quick check that everything runs).
SMOKE = os.environ.get('DL_SMOKE') == '1'

# Images have shape (1, 28, 28): channels first, as PyTorch convolutions expect
mnist = helper.load_mnist(reshape=False)

# Batch Normalization using `nn.BatchNorm1d` and `nn.BatchNorm2d`<a id="example_1"></a>

This version of the network uses PyTorch's `nn` layers for almost everything, and expects you to implement batch normalization using [`nn.BatchNorm1d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) (for the fully connected layer) and [`nn.BatchNorm2d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html) (for the convolutional layers).

We'll use the following function to create fully connected layers in our network. We'll create them with the specified number of neurons and a ReLU activation function.

This version of the function does not include batch normalization.

In [ ]:
"""
DO NOT MODIFY THIS CELL
"""
def fully_connected(num_inputs, num_units):
    """
    Create a fully connectd layer with ReLU activation.

    :param num_inputs: int
        The number of values coming into this layer
    :param num_units: int
        The size of the layer. That is, the number of units, nodes, or neurons.
    :returns nn.Module
        A new fully connected layer
    """
    return nn.Sequential(nn.Linear(num_inputs, num_units), nn.ReLU())

We'll use the following function to create convolutional layers in our network. They are very basic: we're always using a 3x3 kernel, ReLU activation functions, strides of 1x1 on layers with even depths, and strides of 2x2 on layers with odd depths. We aren't bothering with pooling layers at all in this network.

This version of the function does not include batch normalization.

In [ ]:
"""
DO NOT MODIFY THIS CELL
"""
def conv_layer(in_channels, layer_depth):
    """
    Create a convolutional layer with ReLU activation.

    :param in_channels: int
        The number of channels (feature maps) coming into this layer
    :param layer_depth: int
        We'll set the strides and number of feature maps based on the layer's depth in the network.
        This is *not* a good way to make a CNN, but it helps us create this example with very little code.
    :returns nn.Module
        A new convolutional layer
    """
    strides = 2 if layer_depth % 3 == 0 else 1
    return nn.Sequential(nn.Conv2d(in_channels, layer_depth*4, 3, strides, padding=1), nn.ReLU())

**Run the following cell**, along with the earlier cells (to load the dataset and define the necessary functions).

This cell builds the network **without** batch normalization, then trains it on the MNIST dataset. It displays loss and accuracy data periodically while training.

In [ ]:
"""
DO NOT MODIFY THIS CELL
"""
def build_network():
    """Build the network: 19 convolutional layers, one fully connected layer, and a layer with 10 outputs."""
    layers = []
    in_channels, size = 1, 28
    for layer_i in range(1, 20):
        layers.append(conv_layer(in_channels, layer_i))
        in_channels = layer_i * 4
        if layer_i % 3 == 0:
            size = (size - 1) // 2 + 1   # the size of the image after a layer with strides of 2

    # Flatten the output from the convolutional layers
    layers.append(nn.Flatten())

    # Add one fully connected layer
    layers.append(fully_connected(in_channels * size * size, 100))

    # Create the output layer with 1 node for each digit
    layers.append(nn.Linear(100, 10))
    return nn.Sequential(*layers).to(device)


def train(num_batches, batch_size, learning_rate):
    model = build_network()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    def evaluate(images, labels):
        with torch.no_grad():
            logits = model(images)
            return F.cross_entropy(logits, labels).item(), (logits.argmax(dim=1) == labels).float().mean().item()

    # Train and test the network
    for batch_i in range(num_batches):
        batch_xs, batch_ys = mnist.train.next_batch(batch_size)

        # train this batch
        optimizer.zero_grad()
        loss = F.cross_entropy(model(batch_xs), batch_ys)
        loss.backward()
        optimizer.step()

        # Periodically check the validation or training loss and accuracy
        if batch_i % 100 == 0:
            loss, acc = evaluate(mnist.validation.images, mnist.validation.labels)
            print('Batch: {:>2}: Validation loss: {:>3.5f}, Validation accuracy: {:>3.5f}'.format(batch_i, loss, acc))
        elif batch_i % 25 == 0:
            loss, acc = evaluate(batch_xs, batch_ys)
            print('Batch: {:>2}: Training loss: {:>3.5f}, Training accuracy: {:>3.5f}'.format(batch_i, loss, acc))

    # At the end, score the final accuracy for both the validation and test sets
    _, valid_acc = evaluate(mnist.validation.images, mnist.validation.labels)
    print('Final validation accuracy: {:>3.5f}'.format(valid_acc))
    _, test_acc = evaluate(mnist.test.images, mnist.test.labels)
    print('Final test accuracy: {:>3.5f}'.format(test_acc))

    # Score the first 100 test images individually. This won't work if batch normalization isn't implemented correctly.
    correct = 0
    for i in range(100):
        correct += evaluate(mnist.test.images[i:i+1], mnist.test.labels[i:i+1])[1]
    print("Accuracy on 100 samples:", correct/100)
    return test_acc


# Fewer batches in quick-check mode (see SMOKE above)
num_batches = 200 if SMOKE else 800
batch_size = 64
learning_rate = 0.002

train(num_batches, batch_size, learning_rate)

With this many layers, it's going to take a lot of iterations for this network to learn. By the time you're done training these 800 batches, your final test and validation accuracies probably won't be much better than 10%. (It will be different each time, but will most likely be less than 15%.)

Using batch normalization, you'll be able to train this same network to over 90% in that same number of batches.


# Add batch normalization

We've copied the previous three cells to get you started. **Edit these cells** to add batch normalization to the network. For this exercise, you should use `nn.BatchNorm1d` and `nn.BatchNorm2d` to handle most of the math, but you'll need to make a few other changes to your network to integrate batch normalization. You may want to refer back to the lesson notebook to remind yourself of important things, like how a batch normalization layer needs to know whether or not you are performing training or inference.

If you get stuck, you can check out the `Batch_Normalization_Solutions` notebook to see how we did things.

**TODO:** Modify `fully_connected` to add batch normalization to the fully connected layers it creates. Feel free to change the function's parameters if it helps.

In [ ]:
def fully_connected(num_inputs, num_units):
    """
    Create a fully connectd layer with ReLU activation.

    :param num_inputs: int
        The number of values coming into this layer
    :param num_units: int
        The size of the layer. That is, the number of units, nodes, or neurons.
    :returns nn.Module
        A new fully connected layer
    """
    # TODO: add batch normalization (nn.BatchNorm1d) between the linear layer and the ReLU.
    # Batch normalization has its own shift (beta), so the linear layer no longer needs a bias.
    return nn.Sequential(nn.Linear(num_inputs, num_units), nn.ReLU())

**TODO:** Modify `conv_layer` to add batch normalization to the convolutional layers it creates. Feel free to change the function's parameters if it helps.

In [ ]:
def conv_layer(in_channels, layer_depth):
    """
    Create a convolutional layer with ReLU activation.

    :param in_channels: int
        The number of channels (feature maps) coming into this layer
    :param layer_depth: int
        We'll set the strides and number of feature maps based on the layer's depth in the network.
        This is *not* a good way to make a CNN, but it helps us create this example with very little code.
    :returns nn.Module
        A new convolutional layer
    """
    strides = 2 if layer_depth % 3 == 0 else 1
    # TODO: add batch normalization (nn.BatchNorm2d) between the convolution and the ReLU, and drop the bias
    return nn.Sequential(nn.Conv2d(in_channels, layer_depth*4, 3, strides, padding=1), nn.ReLU())

**TODO:** Edit the `train` function to support batch normalization. You'll need to make sure the network knows whether or not it is training: use `model.train()` while training and `model.eval()` when you measure accuracy. That also makes sure it updates and uses its population statistics correctly.

In [ ]:
def train(num_batches, batch_size, learning_rate):
    model = build_network()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    def evaluate(images, labels):
        # TODO: tell the network it is in inference mode
        with torch.no_grad():
            logits = model(images)
            return F.cross_entropy(logits, labels).item(), (logits.argmax(dim=1) == labels).float().mean().item()

    # Train and test the network
    for batch_i in range(num_batches):
        batch_xs, batch_ys = mnist.train.next_batch(batch_size)

        # train this batch
        # TODO: tell the network it is training (batch normalization needs to know)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(batch_xs), batch_ys)
        loss.backward()
        optimizer.step()

        # Periodically check the validation or training loss and accuracy
        if batch_i % 100 == 0:
            loss, acc = evaluate(mnist.validation.images, mnist.validation.labels)
            print('Batch: {:>2}: Validation loss: {:>3.5f}, Validation accuracy: {:>3.5f}'.format(batch_i, loss, acc))
        elif batch_i % 25 == 0:
            loss, acc = evaluate(batch_xs, batch_ys)
            print('Batch: {:>2}: Training loss: {:>3.5f}, Training accuracy: {:>3.5f}'.format(batch_i, loss, acc))

    # At the end, score the final accuracy for both the validation and test sets
    _, valid_acc = evaluate(mnist.validation.images, mnist.validation.labels)
    print('Final validation accuracy: {:>3.5f}'.format(valid_acc))
    _, test_acc = evaluate(mnist.test.images, mnist.test.labels)
    print('Final test accuracy: {:>3.5f}'.format(test_acc))

    # Score the first 100 test images individually. This won't work if batch normalization isn't implemented correctly.
    correct = 0
    for i in range(100):
        correct += evaluate(mnist.test.images[i:i+1], mnist.test.labels[i:i+1])[1]
    print("Accuracy on 100 samples:", correct/100)
    return test_acc


# Fewer batches in quick-check mode (see SMOKE above)
num_batches = 200 if SMOKE else 800
batch_size = 64
learning_rate = 0.002

test_acc = train(num_batches, batch_size, learning_rate)
assert SMOKE or test_acc > 0.9, 'With batch normalization the test accuracy should be over 90%'

With batch normalization, you should now get an accuracy over 90%. Notice also the last line of the output: `Accuracy on 100 samples`. If this value is low while everything else looks good, that means you did not implement batch normalization correctly. Specifically, it means you either did not calculate the population mean and variance while training, or you are not using those values during inference. (If PyTorch raises an error about "more than 1 value per channel", you are evaluating a single image with the network still in training mode.)

# Batch Normalization with your own layer<a id="example_2"></a>

Most of the time you will be able to use the ready-made layers exclusively, but sometimes you may want to work at a lower level. For example, if you ever want to implement a new feature – something new enough that PyTorch does not already include a layer for it, like batch normalization in an LSTM – then you may need to know these sorts of things.

This version of the network uses a batch normalization layer that **you write** using basic tensor operations (`mean`, `var`, arithmetic).

**TODO:** Complete the `BatchNorm` class below. It needs:
- trainable `gamma` (scale, starts at 1) and `beta` (shift, starts at 0), created with `nn.Parameter`,
- the population mean (starts at 0) and variance (starts at 1), stored with `register_buffer` so they are saved with the model but not trained,
- a `forward` that, in training mode (`self.training`), normalizes with the batch's mean and variance and updates the population estimates with a moving average, and otherwise normalizes with the population estimates. It then returns `gamma * x_hat + beta`.

In [ ]:
class BatchNorm(nn.Module):
    """A hand-written batch normalization layer."""

    def __init__(self, num_features, dims, eps=1e-3, momentum=0.01):
        """
        :param num_features: int
            The number of units or feature maps coming out of the layer
        :param dims: tuple of ints
            The dimensions to calculate the batch mean and variance over:
            (0,) for fully connected layers, (0, 2, 3) for convolutional layers
        :param eps: float
            Small constant so we never divide by zero
        :param momentum: float
            How quickly the population estimates follow the batch statistics (1 - decay)
        """
        super().__init__()
        self.dims, self.eps, self.momentum = dims, eps, momentum
        # TODO: create the trainable gamma (ones) and beta (zeros) as nn.Parameter,
        # and the population mean (zeros) and variance (ones) as buffers (see register_buffer)

    def forward(self, x):
        # TODO: in training mode (self.training) normalize with the batch mean and variance
        # and update the population estimates; otherwise use the population estimates.
        # Then scale by gamma and shift by beta. Until you do, this layer just passes x through.
        return x

**TODO:** Modify `fully_connected` to add batch normalization to the fully connected layers it creates, using your `BatchNorm` class. Feel free to change the function's parameters if it helps.

**Optional TODO:** You can run the next cells before you edit them just to see how the network performs without batch normalization. However, the results should be pretty much the same as you saw with the previous example before you added batch normalization.

In [ ]:
def fully_connected(num_inputs, num_units):
    """
    Create a fully connectd layer with ReLU activation.

    :param num_inputs: int
        The number of values coming into this layer
    :param num_units: int
        The size of the layer. That is, the number of units, nodes, or neurons.
    :returns nn.Module
        A new fully connected layer
    """
    # TODO: add batch normalization using your BatchNorm class (dims=(0,)), without a bias and before the ReLU
    return nn.Sequential(nn.Linear(num_inputs, num_units), nn.ReLU())

**TODO:** Modify `conv_layer` to add batch normalization to the convolutional layers it creates.

**Note:** Unlike in the fully connected layer, a convolutional layer _does_ need a slightly different calculation: you need one mean and variance **per feature map**, so you average over the batch, height and width dimensions (`dims=(0, 2, 3)`).

In [ ]:
def conv_layer(in_channels, layer_depth):
    """
    Create a convolutional layer with ReLU activation.

    :param in_channels: int
        The number of channels (feature maps) coming into this layer
    :param layer_depth: int
        We'll set the strides and number of feature maps based on the layer's depth in the network.
        This is *not* a good way to make a CNN, but it helps us create this example with very little code.
    :returns nn.Module
        A new convolutional layer
    """
    strides = 2 if layer_depth % 3 == 0 else 1
    # TODO: add batch normalization using your BatchNorm class (dims=(0, 2, 3)), without a bias and before the ReLU
    return nn.Sequential(nn.Conv2d(in_channels, layer_depth*4, 3, strides, padding=1), nn.ReLU())

The `train` function you edited earlier works without changes here, because `model.train()` and `model.eval()` switch the `self.training` flag of every layer inside the model, including yours. Run the next cell to train the network with your own batch normalization layer.

In [ ]:
test_acc = train(num_batches, batch_size, learning_rate)
assert SMOKE or test_acc > 0.9, 'With batch normalization the test accuracy should be over 90%'

Once again, the model with batch normalization should reach an accuracy over 90%. There are plenty of details that can go wrong when implementing at this low level, so if you got it working - great job! If not, do not worry, just look at the `Batch_Normalization_Solutions` notebook to see what went wrong.